# Data Cleaning Notebook — IPL Ball-by-Ball Analytics
Cleans raw IPL match and delivery files, handles missing values/duplicates/outliers, validates data, engineers features, standardizes teams, converts data types, and exports `Cleaned_Data.csv`.

In [ ]:
from pathlib import Path
import re, pandas as pd, numpy as np
PROJECT_ROOT = Path('..') if Path('../Dataset').exists() else Path('/home/user/Data-Analytics-Project')
DATASET_DIR = PROJECT_ROOT/'Dataset'

deliveries=pd.read_csv(DATASET_DIR/'Raw_Data.csv')
matches=pd.read_csv(DATASET_DIR/'matches_raw.csv')
raw=deliveries.merge(matches.add_prefix('match_'),left_on='match_id',right_on='match_id',how='left')
TEAM_MAP={'Delhi Daredevils':'Delhi Capitals','Kings XI Punjab':'Punjab Kings','Royal Challengers Bangalore':'Royal Challengers Bengaluru','Rising Pune Supergiant':'Rising Pune Supergiants'}
def team(x):
    if pd.isna(x): return 'Unknown'
    x=str(x).strip(); return TEAM_MAP.get(x,x)
clean=raw.copy(); clean.columns=[re.sub(r'[^0-9a-zA-Z]+','_',c.strip()).strip('_').lower() for c in clean.columns]
before=len(clean); clean=clean.drop_duplicates().reset_index(drop=True)
clean['match_date']=pd.to_datetime(clean['match_date'],errors='coerce')
for c in clean.select_dtypes(include='object').columns:
    fill='None' if c in ['extras_type','player_dismissed','dismissal_kind','fielder','match_method'] else 'Unknown'
    clean[c]=clean[c].fillna(fill).astype(str).str.strip()
for c in ['batsman_runs','extra_runs','total_runs','is_wicket','inning','over','ball','match_result_margin','match_target_runs','match_target_overs']:
    if c in clean: clean[c]=pd.to_numeric(clean[c],errors='coerce').fillna(0)
for c in ['batsman_runs','extra_runs','total_runs','is_wicket','inning','over','ball']:
    clean[c]=clean[c].astype(int)
for c in ['batting_team','bowling_team','match_team1','match_team2','match_toss_winner','match_winner']:
    clean[c]=clean[c].apply(team)
clean['expected_total_runs']=clean['batsman_runs']+clean['extra_runs']
clean['run_validation_flag']=np.where(clean['total_runs'].eq(clean['expected_total_runs']),'Valid','Corrected')
clean['total_runs']=clean['expected_total_runs']
clean['match_year']=clean['match_date'].dt.year; clean['match_month']=clean['match_date'].dt.month; clean['match_month_name']=clean['match_date'].dt.strftime('%b'); clean['match_weekday']=clean['match_date'].dt.day_name(); clean['season']=clean['match_season'].astype(str)
clean['legal_ball']=np.where(clean['extras_type'].str.lower().isin(['wides','noballs']),0,1)
clean['over_number']=clean['over']+1; clean['ball_sequence']=clean['over']*10+clean['ball']; clean['match_ball_id']=clean['match_id'].astype(str)+'_'+clean['inning'].astype(str)+'_'+clean['over'].astype(str)+'_'+clean['ball'].astype(str)
clean['phase']=pd.cut(clean['over_number'],bins=[0,6,15,20,100],labels=['Powerplay','Middle Overs','Death Overs','Super Over/Other']).astype(str)
clean['boundary_type']=np.select([clean.batsman_runs.eq(0),clean.batsman_runs.eq(4),clean.batsman_runs.eq(6),clean.batsman_runs.isin([1,2,3])],['Dot Ball','Four','Six','Running Runs'],default='Other')
for name,expr in {'is_boundary':clean.batsman_runs.isin([4,6]),'is_six':clean.batsman_runs.eq(6),'is_four':clean.batsman_runs.eq(4),'is_dot_ball':clean.total_runs.eq(0),'is_extra':clean.extra_runs.gt(0)}.items(): clean[name]=expr.astype(int)
clean['venue_city']=clean['match_city'].replace({'Unknown':'Neutral/Unknown'}); clean['match_label']=clean['match_team1']+' vs '+clean['match_team2']; clean['toss_match_winner_flag']=clean['match_toss_winner'].eq(clean['match_winner']).astype(int); clean['batting_winner_flag']=clean['batting_team'].eq(clean['match_winner']).astype(int)
q1,q3=clean.total_runs.quantile([.25,.75]); clean['runs_outlier_flag']=np.where(clean.total_runs>max(7,q3+1.5*(q3-q1)),'High Run Event','Normal')
print('Rows:',len(clean),'Duplicates removed:',before-len(clean),'Missing after clean:',int(clean.isna().sum().sum()))
keep=['match_id', 'season', 'match_date', 'match_year', 'match_month', 'venue_city', 'match_venue', 'match_winner', 'match_toss_winner', 'match_toss_decision', 'match_result_margin', 'inning', 'batting_team', 'bowling_team', 'over', 'over_number', 'ball', 'batter', 'bowler', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'dismissal_kind', 'legal_ball', 'phase', 'boundary_type', 'is_boundary', 'is_six', 'is_four', 'is_dot_ball', 'is_extra', 'toss_match_winner_flag', 'batting_winner_flag', 'runs_outlier_flag', 'match_player_of_match']
clean=clean[[c for c in keep if c in clean.columns]]
clean.to_csv(DATASET_DIR/'Cleaned_Data.csv',index=False)
clean.head()


## Business interpretation
The cleaned table is a single source of truth for SQL, Power BI, Streamlit, and HTML dashboards. Outliers are retained as valid high-run cricket events and flagged for investigation instead of deleted.